# Troubleshoot Power BI Scanner API

This notebook is a **standalone diagnostic** to inspect exactly what Scanner and fallback Admin APIs return.
It does **not** modify production tables or the main notebook.


In [ ]:
# Cell 1 — Setup (same token/headers as main notebook)
print("[Cell 1] Starting setup...")

import json, time, base64
import requests as http_requests

try:
    pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
except Exception:
    pbi_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")

HEADERS = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type": "application/json",
}
PBI_API = "https://api.powerbi.com/v1.0/myorg"

REQUEST_TIMEOUT_SECONDS = 10
POLL_SECONDS = 3
MAX_POLL_SECONDS = 60

workspace_under_test = None
scan_result = None
dataset_under_test = None
token_info = {}
tenant_metadata_status = "Could not check"

print("Token acquired.")
print("[Cell 1] Setup complete.")



In [ ]:
# Cell 2 — Decode token and print identity + roles
print("[Cell 2] Decoding token...")

try:
    parts = pbi_token.split(".")
    if len(parts) < 2:
        raise ValueError("Token does not look like a JWT.")

    payload = parts[1]
    payload += "=" * (-len(payload) % 4)
    decoded = json.loads(base64.urlsafe_b64decode(payload.encode("utf-8")).decode("utf-8"))

    upn = decoded.get("upn")
    oid = decoded.get("oid")
    appid = decoded.get("appid")
    roles = decoded.get("roles", [])
    scp = decoded.get("scp", "")

    identity_type = "service principal" if appid and not upn else "user"

    token_info = {
        "upn": upn,
        "oid": oid,
        "appid": appid,
        "roles": roles if isinstance(roles, list) else [str(roles)],
        "scp": scp,
        "identity_type": identity_type,
    }

    print(f"Identity (upn or oid): {upn or oid or '(not found)'}")
    print(f"appid: {appid or '(not found)'}")
    print(f"roles: {token_info['roles']}")
    print(f"scp: {scp or '(not found)'}")
    print(f"Identity type guess: {identity_type}")

except Exception as e:
    print(f"[Cell 2] ERROR: {type(e).__name__}: {e}")



In [ ]:
# Cell 3 — Check tenant settings (admin API settings)
print("[Cell 3] Checking /admin/tenantSettings ...")

try:
    url = f"{PBI_API}/admin/tenantSettings"
    resp = http_requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
    print(f"Status: {resp.status_code}")

    if resp.status_code in (401, 403):
        tenant_metadata_status = "Could not check"
        print("Identity likely lacks admin rights for /admin/tenantSettings (401/403).")
    elif resp.status_code != 200:
        tenant_metadata_status = "Could not check"
        print(f"Unexpected response: {resp.text[:500]}")
    else:
        data = resp.json()
        settings_obj = data.get("tenantSettings", data)

        interesting = []
        detailed_flag = None

        if isinstance(settings_obj, dict):
            for k, v in settings_obj.items():
                kl = str(k).lower()
                if "enabledetailedadminapisresponsewithaadmetadata" in kl:
                    detailed_flag = v
                if any(t in kl for t in ["metadata", "admin", "datasource", "enhanced"]):
                    interesting.append((k, v))

        print("enableDetailedAdminApisResponseWithAADMetadata:", detailed_flag)

        if detailed_flag is True:
            tenant_metadata_status = "Enabled"
        elif detailed_flag is False:
            tenant_metadata_status = "Disabled"
        else:
            tenant_metadata_status = "Could not check"

        print("Relevant tenant settings:")
        if interesting:
            for k, v in interesting:
                print(f"- {k}: {json.dumps(v)[:500]}")
        else:
            print("(No matching setting names found in response)")

except Exception as e:
    tenant_metadata_status = "Could not check"
    print(f"[Cell 3] ERROR: {type(e).__name__}: {e}")



In [ ]:
# Cell 4 — Fetch ONE workspace with datasets
print("[Cell 4] Fetching candidate workspaces...")

try:
    url = f"{PBI_API}/admin/groups?$top=100"
    resp = http_requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
    print(f"Status: {resp.status_code}")

    if resp.status_code != 200:
        print(f"Failed to fetch groups: {resp.text[:500]}")
    else:
        groups = resp.json().get("value", [])
        print(f"Groups returned: {len(groups)}")

        workspace_under_test = None
        for g in groups:
            if g.get("type") == "PersonalGroup":
                continue
            workspace_under_test = g
            break

        if workspace_under_test:
            print("Selected workspace:")
            print(f"- id:   {workspace_under_test.get('id')}")
            print(f"- name: {workspace_under_test.get('name')}")
            print(f"- type: {workspace_under_test.get('type')}")
            print("(Dataset presence will be verified in scan result in Cell 5)")
        else:
            print("No non-personal workspace found in first 100 groups.")

except Exception as e:
    print(f"[Cell 4] ERROR: {type(e).__name__}: {e}")



In [ ]:
# Cell 5 — Submit a single scan for that ONE workspace
print("[Cell 5] Submitting one workspace scan...")

try:
    if not workspace_under_test or not workspace_under_test.get("id"):
        raise ValueError("workspace_under_test is not set. Run Cell 4 first.")

    workspace_id = workspace_under_test["id"]
    submit_url = (
        f"{PBI_API}/admin/workspaces/getInfo"
        f"?datasourceDetails=True&getArtifactUsers=True&lineage=True"
        f"&datasetSchema=False&datasetExpressions=False"
    )

    submit_resp = http_requests.post(
        submit_url,
        headers=HEADERS,
        json={"workspaces": [workspace_id]},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    print(f"Submit status: {submit_resp.status_code}")

    if submit_resp.status_code not in (200, 202):
        raise RuntimeError(f"Scan submission failed: {submit_resp.text[:500]}")

    submit_json = submit_resp.json()
    scan_id = submit_json.get("id") or submit_json.get("scanId")
    if not scan_id:
        raise RuntimeError(f"Scan id missing in response: {submit_json}")

    print(f"Scan ID: {scan_id}")

    status_url = f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}"
    result_url = f"{PBI_API}/admin/workspaces/scanResult/{scan_id}"

    start_ts = time.time()
    final_status = "Unknown"
    while True:
        elapsed = int(time.time() - start_ts)
        if elapsed > MAX_POLL_SECONDS:
            break

        st = http_requests.get(status_url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
        st_json = st.json() if st.status_code == 200 else {}
        final_status = st_json.get("status", f"HTTP {st.status_code}")
        print(f"  Poll t={elapsed:02d}s -> {final_status}")

        if str(final_status).lower() in ("succeeded", "failed"):
            break

        time.sleep(POLL_SECONDS)

    if str(final_status).lower() != "succeeded":
        raise RuntimeError(f"Scan did not succeed within {MAX_POLL_SECONDS}s. Last status: {final_status}")

    res = http_requests.get(result_url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
    print(f"Result status: {res.status_code}")
    if res.status_code != 200:
        raise RuntimeError(f"Failed to fetch result: {res.text[:500]}")

    full = res.json()
    workspaces = full.get("workspaces", [])
    if not workspaces:
        raise RuntimeError("scanResult has no workspaces array entries.")

    scan_result = workspaces[0]

    print("Workspace top-level keys:")
    print(sorted(list(scan_result.keys())))

    ds_instances = scan_result.get("datasourceInstances", [])
    print(f"datasourceInstances present: {'Yes' if 'datasourceInstances' in scan_result else 'No'}")
    print(f"datasourceInstances count: {len(ds_instances)}")

except Exception as e:
    print(f"[Cell 5] ERROR: {type(e).__name__}: {e}")



In [ ]:
# Cell 6 — Inspect scan result structure deeply
print("[Cell 6] Deep inspection of scan result...")

try:
    if not scan_result:
        raise ValueError("scan_result is empty. Run Cell 5 first.")

    print("Workspace keys:", sorted(list(scan_result.keys())))
    ws_instances = scan_result.get("datasourceInstances", [])
    ws_instance_ids = set()
    for inst in ws_instances:
        for k in ("datasourceInstanceId", "datasourceId", "id"):
            v = inst.get(k)
            if v:
                ws_instance_ids.add(v)

    datasets = scan_result.get("datasets", [])
    print(f"Datasets found: {len(datasets)}")

    for idx, ds in enumerate(datasets[:3], start=1):
        ds_id = ds.get("id")
        ds_name = ds.get("name")
        print("
" + "-" * 70)
        print(f"Dataset #{idx}: id={ds_id} name={ds_name}")
        print("Dataset keys:", sorted(list(ds.keys())))

        usages = ds.get("datasourceUsages", [])
        ds_level_instances = ds.get("datasourceInstances", [])
        ds_sources = ds.get("datasources", [])

        print(f"datasourceUsages count: {len(usages)}")
        print(f"datasourceInstances count (dataset level): {len(ds_level_instances)}")
        print(f"datasources count: {len(ds_sources)}")

        if usages:
            print("First datasourceUsage:")
            print(json.dumps(usages[0], indent=2))

            usage_id = usages[0].get("datasourceInstanceId") or usages[0].get("datasourceId")
            matches = usage_id in ws_instance_ids if usage_id else False
            print(f"Usage datasource ID: {usage_id}")
            print(f"Matches workspace datasourceInstances IDs: {matches}")

    if ws_instances:
        print("
First workspace datasourceInstance:")
        print(json.dumps(ws_instances[0], indent=2))

except Exception as e:
    print(f"[Cell 6] ERROR: {type(e).__name__}: {e}")



In [ ]:
# Cell 7 — Test fallback API for ONE dataset
print("[Cell 7] Testing fallback /admin/datasets/{id}/datasources ...")

try:
    if not scan_result:
        raise ValueError("scan_result is empty. Run Cell 5 first.")

    datasets = scan_result.get("datasets", [])
    if not datasets:
        raise ValueError("No datasets found in scan_result.")

    dataset_under_test = datasets[0].get("id")
    if not dataset_under_test:
        raise ValueError("First dataset has no ID.")

    print(f"Dataset under test: {dataset_under_test}")

    fb_url = f"{PBI_API}/admin/datasets/{dataset_under_test}/datasources"
    fb_resp = http_requests.get(fb_url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
    print(f"Fallback status: {fb_resp.status_code}")

    if fb_resp.status_code != 200:
        raise RuntimeError(f"Fallback call failed: {fb_resp.text[:1000]}")

    fb_json = fb_resp.json()
    print("Raw fallback JSON response:")
    print(json.dumps(fb_json, indent=2))

    ds_list = fb_json.get("value", [])
    print(f"Datasource objects returned: {len(ds_list)}")

    for i, src in enumerate(ds_list, start=1):
        print("
" + "-" * 70)
        print(f"Fallback datasource #{i}")
        print("keys:", sorted(list(src.keys())))

        id_fields = {
            "datasourceId": src.get("datasourceId"),
            "datasourceInstanceId": src.get("datasourceInstanceId"),
            "id": src.get("id"),
            "gatewayId": src.get("gatewayId"),
            "datasourceType": src.get("datasourceType"),
        }
        print("ID fields:", id_fields)

        conn = src.get("connectionDetails")
        has_conn = bool(conn)
        print(f"connectionDetails populated: {has_conn}")
        if conn:
            print("connectionDetails:", json.dumps(conn))

except Exception as e:
    print(f"[Cell 7] ERROR: {type(e).__name__}: {e}")



In [ ]:
# Cell 8 — Summary / diagnosis
print("[Cell 8] Building summary...")

try:
    roles = token_info.get("roles", []) if isinstance(token_info, dict) else []
    scp = token_info.get("scp", "") if isinstance(token_info, dict) else ""
    has_tenant_read = any(r in ("Tenant.Read.All", "Tenant.ReadWrite.All") for r in roles) or ("Tenant.Read.All" in str(scp))

    datasets = scan_result.get("datasets", []) if isinstance(scan_result, dict) else []
    dataset_count = len(datasets)

    ws_instances = scan_result.get("datasourceInstances", []) if isinstance(scan_result, dict) else []
    ws_inst_count = len(ws_instances)

    usage_count = 0
    for d in datasets:
        usage_count += len(d.get("datasourceUsages", []))

    fallback_count = None
    fallback_have_ids = None
    fallback_id_fields_found = set()
    fallback_has_conn = None

    if dataset_under_test:
        try:
            fb_url = f"{PBI_API}/admin/datasets/{dataset_under_test}/datasources"
            fb_resp = http_requests.get(fb_url, headers=HEADERS, timeout=REQUEST_TIMEOUT_SECONDS)
            if fb_resp.status_code == 200:
                fb_list = fb_resp.json().get("value", [])
                fallback_count = len(fb_list)

                has_any_ids = False
                has_any_conn = False
                for src in fb_list:
                    for k in ("datasourceId", "datasourceInstanceId", "id"):
                        v = src.get(k)
                        if v:
                            has_any_ids = True
                            fallback_id_fields_found.add(k)
                    if src.get("connectionDetails"):
                        has_any_conn = True

                fallback_have_ids = has_any_ids
                fallback_has_conn = has_any_conn
        except Exception as inner_e:
            print(f"Summary fallback recheck warning: {inner_e}")

    print("\n=== TROUBLESHOOT SUMMARY ===")
    print(f"Token identity:                    {token_info.get('identity_type', '(unknown)')}")
    print(f"Has Tenant.Read.All:              {'Yes' if has_tenant_read else 'No'}")
    print(f"Tenant enhanced metadata:          {tenant_metadata_status}")
    print(f"Scanner returns datasets:          {'Yes (' + str(dataset_count) + ')' if dataset_count else 'No'}")
    print(f"datasourceInstances in scan:       {'Yes (' + str(ws_inst_count) + ')' if ws_inst_count else 'No'}")
    print(f"datasourceUsages in scan:          {'Yes (' + str(usage_count) + ')' if usage_count else 'No'}")

    if fallback_count is None:
        print("Fallback API returns datasources:  Could not check")
        print("Fallback datasources have IDs:     Could not check")
        print("Fallback datasources connDetails:  Could not check")
    else:
        print(f"Fallback API returns datasources:  {'Yes (' + str(fallback_count) + ')' if fallback_count else 'No'}")
        print(f"Fallback datasources have IDs:     {'Yes' if fallback_have_ids else 'No'} -> fields found: {sorted(list(fallback_id_fields_found))}")
        print(f"Fallback datasources connDetails:  {'Yes' if fallback_has_conn else 'No'}")

    likely = "D) Everything looks correct → unknown issue, print raw scan sample for manual review"
    if tenant_metadata_status == "Disabled":
        likely = "A) Tenant setting 'Enhance admin APIs' is disabled → enable it in PBI Admin Portal"
    elif not has_tenant_read:
        likely = "B) Identity lacks Tenant.Read.All → grant permission or use admin account"
    elif fallback_count is not None and fallback_count > 0 and not fallback_have_ids:
        likely = "C) Fallback API returns datasources without ID fields → need different ID extraction"

    print("\nLIKELY ROOT CAUSE:")
    print(f"  {likely}")

except Exception as e:
    print(f"[Cell 8] ERROR: {type(e).__name__}: {e}")

